# Lab 1: An FNO on Allen–Cahn

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maximilianherde/ricam-foundation-models-for-pdes/blob/main/notebooks/01_operator_learning_fno.ipynb)

**RICAM Training School, Linz, 8 October 2026**

In this lab you implement a Fourier neural operator (FNO), train it on the
Allen–Cahn dataset of the Poseidon paper, and test it in three ways:

- **A.** evaluate the trained model on a finer grid than it was trained on;
- **B.** apply it repeatedly to reach later times (autoregressive rollout);
- **C.** compare this with a model that is told how far ahead to predict.

Set the Colab runtime to a GPU: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
import math, os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
from tqdm.auto import tqdm

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
torch.manual_seed(0)
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


def sync():
    # GPU calls are asynchronous; wait for them before reading the clock.
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


N_TRAIN, N_TEST = 128, 240        # trajectories
WIDTH, MODES, N_LAYERS = 32, 16, 4
EPOCHS, BATCH = 60, 32

## The problem

The Allen–Cahn equation

$$u_t = \Delta u - \varepsilon^2\, u\,(u^2 - 1), \qquad \varepsilon = 220,$$

on the unit square. The reaction term drives $u$ towards the two stable states
$\pm 1$; diffusion smooths the interfaces between them. Over time, regions of
$+1$ and $-1$ grow and merge.

## The data

We use the released dataset (`camlab-ethz/ACE` on Hugging Face). It holds
15,000 trajectories in four files of 3,750 each. The paper uses the last 240
trajectories, the end of `solution_3.nc`, as its test set; we do the same. For
training we take the first 128 trajectories of `solution_0.nc`.

**Which snapshots.** Like the paper, we use only every other snapshot,
$0, 2, \dots, 14$ (in `scOT`: `time_step_size = 2`, `max_num_time_steps = 7`).
From here on we number these 8 snapshots $k = 0, \dots, 7$; one step in $k$ is
$6\cdot10^{-6}$ in time. The paper evaluates the map from
$k = 0$ to $k = 7$ (snapshot 14).

The files are large but stored one snapshot at a time, so we can read only the
data we need over HTTP (about 70 MB for training, 130 MB for testing).

In [ ]:
try:
    import fsspec, h5py
except ImportError:
    !pip install -q fsspec h5py
    import fsspec, h5py

URL = "https://huggingface.co/datasets/camlab-ethz/ACE/resolve/main/solution_{}.nc"


def load_ace(file, start, stop, cache="ace_cache"):
    # snapshots 0, 2, ..., 14 only
    path = os.path.join(cache, f"ace{file}_{start}_{stop}_even.npy")
    if os.path.exists(path):
        return np.load(path)
    os.makedirs(cache, exist_ok=True)
    with fsspec.open(URL.format(file), "rb", block_size=2**22) as f, h5py.File(f, "r") as h:
        data = h["solution"][start:stop, 0:15:2]
    np.save(path, data)
    return data


train_raw = load_ace(0, 0, N_TRAIN)
test_raw = load_ace(3, 3750 - N_TEST, 3750)
N_T = train_raw.shape[1]                      # 8 snapshots, k = 0..7
print("train", train_raw.shape, "  test", test_raw.shape)

In [ ]:
def movie(fields, titles, labels, ranges=None):
    # fields: list of arrays of shape (frames, H, W), shown side by side;
    # labels: one title per frame
    n = len(fields)
    ranges = ranges or [(-1, 1)] * n
    fig, axes = plt.subplots(1, n, figsize=(2.7 * n, 2.9))
    axes = np.atleast_1d(axes)
    ims = []
    for ax, f, t, (lo, hi) in zip(axes, fields, titles, ranges):
        ims.append(ax.imshow(f[0], cmap="gist_ncar", vmin=lo, vmax=hi))
        ax.set_title(t, fontsize=9); ax.axis("off")
    label = fig.suptitle(labels[0])

    def frame(k):
        for im, f in zip(ims, fields):
            im.set_data(f[k])
        label.set_text(labels[k])
        return ims

    anim = animation.FuncAnimation(fig, frame, frames=len(fields[0]), interval=400)
    plt.close(fig)
    return HTML(anim.to_jshtml())


FRAMES = [f"k = {k}  (snapshot {2 * k})" for k in range(N_T)]
movie([train_raw[j] for j in range(3)], [f"training trajectory {j}" for j in range(3)], FRAMES)

### Normalisation and error metric

As in Poseidon, every field is standardised with the dataset mean and standard
deviation, $\tilde u = (u - \mu)/\sigma$ with $\mu = 0.00248$,
$\sigma = 0.6535$. The model sees $\tilde u$, and we report errors on
$\tilde u$ as well, so the numbers are comparable to the paper's. One could
equally report them on the physical field $u$; the values differ slightly,
because a relative error is not invariant under the shift by $\mu$.

The metric is the relative $L^1$ error per sample,
$\lVert \hat u - u \rVert_1 / \lVert u \rVert_1$, and we report the median
over the test set.

In [ ]:
MEAN, STD = 0.002484262, 0.65351176           # Poseidon's constants for ACE

train128 = torch.from_numpy((train_raw - MEAN) / STD).to(device)
test128 = torch.from_numpy((test_raw - MEAN) / STD).to(device)


def rel_l1(pred, true):
    # one number per sample
    return (pred - true).abs().flatten(1).sum(1) / true.abs().flatten(1).sum(1)

## The FNO

A neural operator lifts the input to a wider channel space, applies a few
layers of the form

$$v_{\ell+1}(x) = \sigma\Big(\int \kappa(x,y)\,v_\ell(y)\,dy + W v_\ell(x)\Big),$$

and projects back. The FNO takes $\kappa(x,y) = \kappa(x-y)$, a convolution,
so the integral becomes a multiplication in Fourier space:
$\mathcal{F}^{-1}\big(R \cdot \mathcal{F} v_\ell\big)$, with a learned complex
matrix $R_k$ for each wavenumber $k$. Only the lowest `MODES` wavenumbers in
each direction are kept; all higher ones are set to zero.

**Exercises 1 and 2** ask you to write the two classes. Each exercise cell
raises `NotImplementedError` until you fill it in. Each exercise has a hint
that you can open if you are stuck. Solutions are handed out after the lab.

### Exercise 1: `SpectralConv2d`

Implement `forward`. It maps `x` of shape `(B, c_in, H, W)` to
`(B, c_out, H, W)`: transform to Fourier space (`torch.fft.rfft2`), multiply
the retained low modes by the learned complex weights, which mix the channels
($R_k$ is a `c_in × c_out` matrix for each mode $k$), set all other modes to
zero, and transform back (`torch.fft.irfft2`). The layer must work for any
`H` and `W`: later we apply it on other grids.

<details><summary><b>Hint</b></summary>

`rfft2` keeps only the non-negative frequencies in the last axis, but both
signs in the first axis. The low modes are therefore `[:m1, :m2]` *and*
`[-m1:, :m2]`, which is why there are two weight tensors. Pass `s=(H, W)` to
`irfft2`.

</details>

In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, c_in, c_out, m1, m2):
        super().__init__()
        self.c_out, self.m1, self.m2 = c_out, m1, m2
        scale = 1.0 / (c_in * c_out)
        self.w_lo = nn.Parameter(scale * torch.rand(c_in, c_out, m1, m2, dtype=torch.cfloat))
        self.w_hi = nn.Parameter(scale * torch.rand(c_in, c_out, m1, m2, dtype=torch.cfloat))

    def forward(self, x):                       # x: (B, c_in, H, W)
        B, _, H, W = x.shape
        m1, m2 = min(self.m1, H // 2), min(self.m2, W // 2 + 1)
        raise NotImplementedError("Exercise 1")

### Exercise 2: `FNO2d`

Build the network from the template above: lift the input to `width`
channels, apply `n_layers` Fourier layers
$v \mapsto \sigma\big(\mathcal{F}^{-1}(R \cdot \mathcal{F}v) + Wv\big)$ with
`SpectralConv2d`, and project back to one output channel with a small
two-layer network. Lifting, $W$ and projection act pointwise. The input is the
single channel $\tilde u$ (two channels for the model with a lead time later).

<details><summary><b>Hint</b></summary>

A pointwise linear map on a grid is a $1\times1$ convolution,
`nn.Conv2d(c_in, c_out, 1)`. Leave out the activation after the last Fourier
layer.

</details>

In [ ]:
class FNO2d(nn.Module):
    def __init__(self, in_channels=1, width=WIDTH, modes=MODES, n_layers=N_LAYERS):
        super().__init__()
        raise NotImplementedError("Exercise 2")

    def forward(self, x):                       # (B, in_channels, H, W) -> (B, 1, H, W)
        raise NotImplementedError("Exercise 2")

## Training the one-step map

The **one-step map** takes $u_k$ to $u_{k+1}$, i.e. it advances the solution
by $6\cdot10^{-6}$ in time (two snapshots of the dataset). Each trajectory
gives 7 such input–output pairs.

We train on a $64\times64$ version of the data, obtained by keeping every
second grid point, and keep the $128\times128$ data for experiment A. The test
pairs come from the 240 test trajectories, so no test trajectory is seen in
training.

In [ ]:
def coarsen(t, factor):
    return t[..., ::factor, ::factor].contiguous()


def pairs(t, lead=1):
    # (n, 8, H, W) -> inputs u_k and targets u_{k+lead}, each (n * (8 - lead), 1, H, W),
    # ordered by start k first, then by trajectory
    H, W = t.shape[-2:]
    x = t[:, :N_T - lead].transpose(0, 1).reshape(-1, 1, H, W)
    y = t[:, lead:].transpose(0, 1).reshape(-1, 1, H, W)
    return x, y


train64, test64 = coarsen(train128, 2), coarsen(test128, 2)
X_tr, Y_tr = pairs(train64)
X_te, Y_te = pairs(test64)
print(f"{len(X_tr)} training pairs, {len(X_te)} test pairs")

def model_input(u, lead=None):
    # (B, 1, H, W) -> u itself, or (B, 2, H, W) = [u, lead / 7] if a lead time is given
    if lead is None:
        return u
    b, _, h, w = u.shape
    lead = torch.as_tensor(lead, dtype=u.dtype, device=device).reshape(-1, 1, 1, 1)
    return torch.cat([u, (lead / (N_T - 1)).expand(b, 1, h, w)], dim=1)



class LiveLoss:
    # Plots the training loss while training. Call update(loss) every step and
    # close() at the end; the plot shows the mean over every `every` steps.
    def __init__(self, title="", every=20):
        self.every, self.steps, self.values = every, [], []
        self.window, self.n = 0.0, 0
        self.fig, self.ax = plt.subplots(figsize=(5, 2.6))
        self.title = title
        self.handle = display(self.fig, display_id=True)
        plt.close(self.fig)

    def update(self, loss):
        self.window = self.window + loss.detach()
        self.n += 1
        if self.n % self.every == 0:
            self.steps.append(self.n)
            self.values.append(self.window.item() / self.every)
            self.window = 0.0
            self.draw()

    def draw(self):
        self.ax.clear()
        self.ax.semilogy(self.steps, self.values)
        self.ax.set_xlabel("step"); self.ax.set_ylabel("training loss")
        self.ax.set_title(self.title, fontsize=9)
        self.fig.tight_layout()
        self.handle.update(self.fig)

    def close(self):
        if self.steps:
            self.draw()



def train(model, batch_loss, n_steps, lr=2e-3, desc=""):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, n_steps)       # lr decays from lr to 0
    model.train()
    live = LiveLoss(desc)
    for step in tqdm(range(n_steps), desc=desc):
        loss = batch_loss()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step(); sched.step()
        live.update(loss)
    live.close()


@torch.no_grad()
def errors(model, X, Y, lead=None, batch=64):
    model.eval()
    return torch.cat([rel_l1(model(model_input(X[i:i + batch], lead)), Y[i:i + batch])
                      for i in range(0, len(X), batch)])

In [ ]:
model = FNO2d().to(device)
print(f"{sum(p.numel() for p in model.parameters()):,} parameters")

N_STEPS_TRAIN = EPOCHS * math.ceil(len(X_tr) / BATCH)


def one_step_loss():
    idx = torch.randint(0, len(X_tr), (BATCH,), device=device)
    return rel_l1(model(model_input(X_tr[idx])), Y_tr[idx]).mean()


train(model, one_step_loss, N_STEPS_TRAIN, desc="one-step")

err64 = errors(model, X_te, Y_te)
print(f"median relative L1 error, one step, 64x64: {err64.median():.4f}")

## Experiment A: other resolutions

The weights of an FNO act on Fourier modes, not on grid points, so the same
model can be applied on any grid without changes. What we would like is that
the model commutes with a change of resolution: its output on a finer grid
should be the finer sampling of the function it predicts on the coarse grid.
This is an equivariance, not an invariance. The output does change with the
grid; it should change only by being sampled differently.

We evaluate the model trained at $64\times64$ on the test data at
$16\times16$, $32\times32$, $64\times64$ and $128\times128$ (obtained by
subsampling the $128\times128$ data) and plot the error against the
resolution.

In [ ]:
RESOLUTIONS = [16, 32, 64, 128]
err_by_res = {}
for R in RESOLUTIONS:
    X_R, Y_R = pairs(coarsen(test128, 128 // R))
    err_by_res[R] = errors(model, X_R, Y_R, batch=32).median().item()
    print(f"evaluated at {R:3d}x{R:<3d}: median relative L1 {err_by_res[R]:.4f}")

fig, ax = plt.subplots(figsize=(4.8, 3.2))
ax.plot(RESOLUTIONS, [err_by_res[R] for R in RESOLUTIONS], "o-")
ax.axvline(64, color="gray", ls="--", lw=1, label="training resolution")
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xticks(RESOLUTIONS); ax.set_xticklabels([f"{R}²" for R in RESOLUTIONS])
ax.set_xlabel("evaluation resolution"); ax.set_ylabel("median relative $L^1$")
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

On coarser grids the input itself has lost information, so the error has to
grow. On finer grids it can grow too, because of aliasing. Each layer applies a
pointwise linear map and a GELU, and these create frequencies above the
truncation. On the $64\times64$ training grid, frequencies above its Nyquist
frequency are folded back onto lower ones; on a finer grid they are not. So the
same weights compute a slightly different function on each grid. See
Bartolucci et al., *Representation Equivalent Neural Operators* (NeurIPS 2023);
the Convolutional Neural Operator (Raonic et al., NeurIPS 2023) is designed to
avoid this.

## Experiment B: autoregressive rollout

To reach $u_k$ from $u_0$ with the one-step model, apply it $k$ times, each
time to its own previous output. We compare the error of this rollout with the
error of a single step from the true $u_{k-1}$. The difference comes from
feeding the model its own predictions, which are not exact samples from the
training distribution.

In [ ]:
@torch.no_grad()
def rollout(model, u0, n_steps):
    model.eval()
    u, out = u0, []
    for _ in range(n_steps):
        u = model(model_input(u))
        out.append(u)
    return torch.cat(out, dim=1)


preds = rollout(model, test64[:, :1], N_T - 1)
err_rollout = np.array([rel_l1(preds[:, k - 1], test64[:, k]).median().item()
                        for k in range(1, N_T)])
err_onestep = err64.reshape(N_T - 1, N_TEST).median(1).values.cpu().numpy()

steps = np.arange(1, N_T)
fig, ax = plt.subplots(figsize=(5, 3))
ax.semilogy(steps, err_rollout, "o-", ms=3, label="rollout from snapshot 0")
ax.semilogy(steps, err_onestep, "s-", ms=3, label="one step from the true snapshot")
ax.set_xlabel("$k$"); ax.set_ylabel("median relative $L^1$")
ax.legend(frameon=False)
plt.show()
print(f"rollout error at k = 7 (snapshot 14): {err_rollout[-1]:.4f}")

The rollout for one test trajectory, next to the true solution (both in
physical units, on the $64\times64$ grid):

In [ ]:
j = 0                                            # test trajectory to show
true_u = (test64[j] * STD + MEAN).cpu().numpy()
fno_u = np.concatenate([true_u[:1], (preds[j] * STD + MEAN).cpu().numpy()])   # starts from the true u_0
movie([true_u, fno_u, np.abs(fno_u - true_u)],
      ["true solution", "FNO rollout", "|error|"], FRAMES,
      ranges=[(-1, 1), (-1, 1), (0, 0.5)])

## Experiment C: predicting with a lead time

Instead of a fixed step, train one model $G(u_k, \ell) \approx u_{k+\ell}$ that
takes the lead time $\ell \in \{1, \dots, 7\}$ as an input. Then $u_7$ can be
reached from $u_0$ in a single call, or in a few larger steps.

**How the lead time enters the model.** As a second input channel that is
constant over the domain, with value $\ell/7$ (see `model_input` above). So
the model is `FNO2d(in_channels=2)`. Poseidon uses a different mechanism: the
lead time scales and shifts the layer norms in every block.

**Training data.** Every pair $(k, k+\ell)$ with $0 \le k < k+\ell \le 7$
from a trajectory is a training example: 28 per trajectory instead of 7.
Poseidon calls this *all2all* training. The model gets the same number of
gradient steps as the one-step model.

### Exercise 3: the all2all batch

Draw `BATCH` random triples (trajectory $i$, start $k$, lead $\ell$) with
$\ell \ge 1$ and $k + \ell \le 7$, and return the mean relative $L^1$ loss of
`lead_model` on them. The training data is `train64`, of shape
`(N_TRAIN, 8, 64, 64)`.

<details><summary><b>Hint</b></summary>

Draw $k$ first and then $\ell$ uniformly from $1, \dots, 7 - k$. `model_input`
accepts a tensor with one lead time per sample.

</details>

In [ ]:
lead_model = FNO2d(in_channels=2).to(device)
T_MAX = N_T - 1                                 # 7

In [ ]:
def all2all_loss():
    raise NotImplementedError("Exercise 3")

In [ ]:
train(lead_model, all2all_loss, N_STEPS_TRAIN, desc="all2all")

### Reaching $u_7$

Starting from $u_0$ of every test trajectory, we reach $u_7$ (snapshot 14) in
several ways and time each one:

- **direct**: one call with $\ell = 7$;
- **4 + 3** and **2 + 2 + 2 + 1**: a few larger steps;
- **7 × 1**: the lead-time model with $\ell = 1$, seven times;
- **one-step model**: the model from before, seven times.

In [ ]:
@torch.no_grad()
def reach_7(m, schedule):
    m.eval()
    u = test64[:, :1]
    for lead in schedule:
        u = m(model_input(u, lead))
    return u


def timed(fn, repeats=3):
    fn(); sync()                                # warm-up
    t0 = time.time()
    for _ in range(repeats):
        out = fn()
    sync()
    return out, (time.time() - t0) / repeats


schedules = [("direct", lead_model, [7]),
             ("4 + 3", lead_model, [4, 3]),
             ("2 + 2 + 2 + 1", lead_model, [2, 2, 2, 1]),
             ("7 x 1", lead_model, [1] * 7),
             ("one-step model", model, [None] * 7)]

target = test64[:, -1]
results = []
print(f"{'schedule':16} {'calls':>5} {'median rel. L1':>15} {'time (ms)':>10}")
for name, m, sched in schedules:
    out, dt = timed(lambda m=m, s=sched: reach_7(m, s))
    e = rel_l1(out[:, 0], target).median().item()
    results.append((name, len(sched), e, dt))
    print(f"{name:16} {len(sched):5d} {e:15.4f} {1000 * dt:10.1f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
for name, calls, e, dt in results:
    ax.plot(1000 * dt, e, "o")
    ax.annotate(name, (1000 * dt, e), textcoords="offset points", xytext=(5, 3), fontsize=8)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel(f"time for {N_TEST} test trajectories (ms)")
ax.set_ylabel("median relative $L^1$ at $k = 7$")
plt.show()

## References

Li et al., *Fourier Neural Operator for Parametric PDEs*, ICLR
2021 · Kovachki et al., *Neural Operator*, JMLR 2023 · Bartolucci et al.,
*Representation Equivalent Neural Operators*, NeurIPS 2023 · Raonic et al.,
*Convolutional Neural Operators*, NeurIPS 2023 · Herde et al., *Poseidon*,
NeurIPS 2024 (arXiv:2405.19101).